# 5-Layer Autonomous GenAI-Powered Cloud Defense Architecture
## Large-Scale Research & Experimental Pipeline

This notebook contains the fully optimized, dissertation-grade pipeline. 
**Upgrades Included:**
- Scaled dataset loaders (up to 100k+ samples with `pin_memory` and `prefetching`)
- **FP16 Mixed Precision (AMP)** for significantly faster multi-epoch training on T4/L4 GPUs
- **Cosine Annealing Schedulers & Gradient Clipping** for convergence stability
- **Multi-Seed Ablation Framework** computing strictly validated Mean ± Std across runs
- **MLflow Tracking** embedded directly for transparency

In [ ]:
# Environment Setup
!git clone https://github.com/tushars00d/GAICS_v2.git || echo "Already cloned"
%cd GAICS_v2
!pip install -r requirements.txt -q

### Phase 1: High-Performance Generative Pre-training (Tabular DDPM)
This scales up data generation using PyTorch's Mixed Precision and outputs the loss trajectory.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from training.train_ddpm import train_ddpm

sns.set_theme(style="whitegrid")
ddpm_model, ddpm_history = train_ddpm("configs/default.yaml")

# Plot DDPM Loss Trajectory
plt.figure(figsize=(10, 4))
plt.plot(ddpm_history['loss'], label="DDPM MSE Loss", color='purple', linewidth=2)
plt.title("Tabular DDPM Generative Pre-training Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

### Phase 2: Robust Attention IDS Training
Training the detection model using Focal Loss, Cosine Schedulers, and Gradient Clipping.

In [ ]:
from training.train_ids import train_ids

ids_model, ids_history = train_ids("configs/default.yaml")

# Plot IDS Training Trajectory
fig, ax1 = plt.subplots(figsize=(10, 4))

ax1.set_xlabel('Epochs')
ax1.set_ylabel('Focal Loss', color='tab:red')
ax1.plot(ids_history['train_loss'], color='tab:red', label='Train Loss')
ax1.tick_params(axis='y', labelcolor='tab:red')

ax2 = ax1.twinx()
ax2.set_ylabel('Macro F1 Score', color='tab:blue')
ax2.plot(ids_history['val_f1'], color='tab:blue', label='Val Macro F1')
ax2.tick_params(axis='y', labelcolor='tab:blue')

plt.title("Attention IDS Training Curve")
fig.tight_layout()
plt.show()

### Phase 3: Rigorous Ablation Studies (5 Seeds)
Evaluates **Clean Performance** vs. **Adversarial (FGSM)** vs. **DDPM Purified**.
Generates detailed statistics (Mean ± Std) and Confusion Matrices.

In [ ]:
from evaluation.benchmark import run_ablation_studies
import IPython.display as display

results = run_ablation_studies("configs/default.yaml")

print("\nVisualizing Confusion Matrices from the final seed:\n")
display.display(display.Image(filename="plots/cm_clean.png"))
display.display(display.Image(filename="plots/cm_purified.png"))

### Phase 4: Autonomous SOAR Agent Decision Trace
Passing an incident to the LangChain RAG pipeline for resolution mapping.

In [ ]:
import yaml
from agents.agents import AutonomousResponseAgent

with open("configs/default.yaml", "r") as f:
    config = yaml.safe_load(f)

agent = AutonomousResponseAgent(config)
decision = agent.process_incident([0.5, -0.2, 0.8, 1.2])
print("Final Action Execution trace:\n", decision)